# 12 — 床反力から関節トルクへ

立脚のGRF変換と遊脚の追従制御が、同じ脚トルク出力へ合流する仕組みを確認します。

**前提**: `11_acados_and_receding_horizon.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 床反力から関節トルクへの変換を上流PyMPCと同じ実行条件で確認します。
# 目的: ワークスペースとPyMPCの場所を確定し、後続セルの実行条件を再現可能にします。
# OSに依存しないパス演算を行うためPathを読み込む。
from pathlib import Path
# 環境変数の設定にos、モジュール検索パスの設定にsysを使う。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化する。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけリポジトリルートへ移る。
if ROOT.name == "notebook_pympc":
    # 外部実装をROOT基準で参照できるよう親ディレクトリを採用する。
    ROOT = ROOT.parent
# 上流Quadruped-PyMPCの配置先をROOTから組み立てる。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動場所のまま実験を進めないよう実装の存在を検証する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同名モジュールの取り違えを防ぐため検索パス未登録時だけ処理する。
if str(PYMPC_ROOT) not in sys.path:
    # 現行リポジトリの実装を最優先でimportするため先頭へ追加する。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物の探索基準を未設定時だけ上流同梱ディレクトリへ合わせる。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画できるよう未設定時はEGLを選ぶ。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実験が参照するワークスペースを目視確認できるよう表示する。
print("workspace :", ROOT)
# 上流実装の参照先を目視確認できるよう表示する。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 立脚脚

virtual workより

\[
\tau_{stance}=-J^TF^{cmd}
\]

符号は「ロボットが地面へ及ぼす力」と「地面がロボットへ及ぼす力」の定義で変わります。
現行実装の符号を正にしてください。

## 遊脚脚

Cartesian PDとfeedback linearizationを使い、足位置軌道を追従します。
contact scheduleが立脚・遊脚のどちらの式を使うかを選びます。

In [2]:
# 背景: 立脚トルクは仮想仕事からτ_stance=-J_W(q)^T F_Wで計算されます。
# 目的: world座標の1脚GRFを3関節トルクへ写像し、shape・単位・符号を確認します。
# 行列積とベクトルshapeの確認にNumPyを使う。
import numpy as np

# 1脚の並進Jacobianをshape (3 foot axes, 3 joints)で定義する。
# 第1行はworld x方向の足速度に対する3関節の寄与 [m/rad] を表す。
# 第2行はworld y方向の足速度に対する3関節の寄与 [m/rad] を表す。
# 第3行はworld z方向の足速度に対する3関節の寄与 [m/rad] を表す。
J = np.array([[0.0, 0.20, 0.15],
              [0.18, 0.00, 0.00],
              [0.00, 0.12, 0.22]])  # [m/rad], shape (3 foot axes, 3 joints)

# MPC出力のworld座標GRFを[Fx,Fy,Fz]=[10,0,50] Nで定義する。
F = np.array([10., 0., 50.])  # [N], shape (3,)

# 仮想仕事δW=F^Tδp=F^TJδqと上流の作用方向定義からτ=-J^TFを計算する。
tau = -J.T @ F  # [N m], shape (3 joints,)
# 行列積の前提となるJとFのshapeを表示して軸の取り違えを検出する。
print("J shape:", J.shape, "F shape:", F.shape)
# 得られた3関節トルクをN m単位で表示する。
print("stance torque:", tau, "N m")
# 出力が1脚3関節に対応するshape (3,)であることを検証する。
assert tau.shape == (3,)

J shape: (3, 3) F shape: (3,)
stance torque: [  0.   -8.  -12.5] N m


In [3]:
# 背景: 計算した関節トルクはアクチュエータ上限内でも、安全余裕を持つsoft limitで制限します。
# 目的: Go2の3関節上限の90%でτをclipし、どの関節が飽和したかを確認します。
# 各関節の絶対トルク上限 [N m]へ安全係数0.9を掛けてsoft limitを作る。
limits = np.array([23.7, 23.7, 45.4]) * 0.9
# 各τ_iを対応する[-limit_i,+limit_i]へ要素ごとに制限する。
clipped = np.clip(tau, -limits, limits)
# 3関節のsoft limitをN m単位で表示する。
print("soft limits:", limits)
# Plantへ送れる範囲に制限したトルクをN m単位で表示する。
print("clipped    :", clipped)
# |τ_i|>limit_iを関節ごとの飽和フラグとして表示する。
print("saturated? :", np.abs(tau) > limits)

soft limits: [21.33 21.33 40.86]
clipped    : [  0.   -8.  -12.5]
saturated? : [False False False]


トルク飽和は原因ではなく結果です。大きいGRF、悪い足位置によるJacobian、
高い遊脚ゲイン、短いswing時間のいずれでも起きます。
飽和した脚・位相・寄与項を分解してから調整します。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。